1. IDA*

In [3]:
ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def manhattan_distance(current, goal):
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = current[r][c]
            if val != 0:
                target_r, target_c = goal_pos[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def print_matrix(state):
    for row in state:
        print("  ", [x if x != 0 else " " for x in row])
    print()

def idfs_astar(current, goal, g_cost, f_limit, path, visited):
    h_cost = manhattan_distance(current, goal)
    f_cost = g_cost + h_cost

    if current == goal:
        return path, f_cost
    if f_cost > f_limit:
        return None, f_cost

    visited.add(current)
    x, y = find_zero(current)
    min_cutoff = float('inf')

    for move, (dx, dy), move_name in ACTIONS:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            neighbor = swap(current, x, y, nx, ny)
            if neighbor not in visited:
                result, next_f = idfs_astar(neighbor, goal, g_cost + 1, f_limit, path + [move], visited)

                if result is not None:
                    return result, next_f
                if next_f < min_cutoff:
                    min_cutoff = next_f

    visited.remove(current)  # Backtrack chuyên sâu
    return None, min_cutoff

def run_ida_star(start, goal):
    print("THUẬT TOÁN IDA* ")
    f_limit = manhattan_distance(start, goal)
    loop_count = 1

    while True:
        print(f"Vòng lặp {loop_count}: Thử nghiệm với giới hạn f_limit = {f_limit}")
        visited = set()
        result, next_f = idfs_astar(start, goal, 0, f_limit, [], visited)

        if result is not None:
            print(f"=> Thành công tại vòng lặp {loop_count}!\n")
            return result
        if next_f == float('inf'):
            return None

        f_limit = next_f
        loop_count += 1

if __name__ == "__main__":
    START_STATE = (
        (1, 2, 3),
        (4, 0, 6),
        (7, 5, 8)
    )

    GOAL_STATE = (
        (1, 2, 3),
        (4, 5, 6),
        (7, 8, 0)
    )

    ida_path = run_ida_star(START_STATE, GOAL_STATE)
    print(f"Kết quả đường đi IDA*: {ida_path}")
    print("-" * 50)

THUẬT TOÁN IDA* 
Vòng lặp 1: Thử nghiệm với giới hạn f_limit = 2
=> Thành công tại vòng lặp 1!

Kết quả đường đi IDA*: ['D', 'R']
--------------------------------------------------


2. Simple Hill C

In [4]:
ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def manhattan_distance(current, goal):
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = current[r][c]
            if val != 0:
                target_r, target_c = goal_pos[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def print_matrix(state):
    for row in state:
        print("  ", [x if x != 0 else " " for x in row])
    print()

def idfs_astar(current, goal, g_cost, f_limit, path, visited):
    h_cost = manhattan_distance(current, goal)
    f_cost = g_cost + h_cost

    if current == goal:
        return path, f_cost
    if f_cost > f_limit:
        return None, f_cost

    visited.add(current)
    x, y = find_zero(current)
    min_cutoff = float('inf')

    for move, (dx, dy), move_name in ACTIONS:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            neighbor = swap(current, x, y, nx, ny)
            if neighbor not in visited:
                result, next_f = idfs_astar(neighbor, goal, g_cost + 1, f_limit, path + [move], visited)

                if result is not None:
                    return result, next_f
                if next_f < min_cutoff:
                    min_cutoff = next_f

    visited.remove(current)
    return None, min_cutoff

def run_simple_hill_climbing(start, goal):
    print("THUẬT TOÁN SIMPLE HILL CLIMBING")
    current = start
    path = []
    curr_h = manhattan_distance(current, goal)

    step = 0
    print(f"Bước {step} (Trạng thái START): h = {curr_h}")
    print_matrix(current)

    while current != goal:
        x, y = find_zero(current)
        neighbor_moved = False

        # Thử từng hướng theo độ ưu tiên L -> R -> U -> D
        for move, (dx, dy), move_name in ACTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                neighbor = swap(current, x, y, nx, ny)
                neighbor_h = manhattan_distance(neighbor, goal)

                # Leo đồi đơn giản: Thấy con đầu tiên tốt hơn hoặc bằng là đi luôn
                if neighbor_h < curr_h:
                    current = neighbor
                    curr_h = neighbor_h
                    path.append(move)
                    neighbor_moved = True

                    step += 1
                    print(f"Bước {step}: Đi hướng [{move}] {move_name} -> h giảm xuống = {curr_h}")
                    print_matrix(current)
                    break  # Ngắt ngay lập tức để di chuyển sang nhánh mới, bỏ qua các hướng còn lại

        # Nếu đã check hết cả 4 hướng mà không hướng nào tối ưu hơn trạng thái hiện tại
        if not neighbor_moved:
            print("KẸT CỤC BỘ! Thuật toán dừng lại vì không tìm được ô nào tốt hơn.")
            return None

    return path

if __name__ == "__main__":
    START_STATE = (
        (1, 2, 3),
        (4, 0, 6),
        (7, 5, 8)
    )

    GOAL_STATE = (
        (1, 2, 3),
        (4, 5, 6),
        (7, 8, 0)
    )

    hill_path = run_simple_hill_climbing(START_STATE, GOAL_STATE)
    print(f"Kết quả đường đi Hill Climbing: {hill_path}")

THUẬT TOÁN SIMPLE HILL CLIMBING
Bước 0 (Trạng thái START): h = 2
   [1, 2, 3]
   [4, ' ', 6]
   [7, 5, 8]

Bước 1: Đi hướng [D] XUỐNG -> h giảm xuống = 1
   [1, 2, 3]
   [4, 5, 6]
   [7, ' ', 8]

Bước 2: Đi hướng [R] SANG PHẢI -> h giảm xuống = 0
   [1, 2, 3]
   [4, 5, 6]
   [7, 8, ' ']

Kết quả đường đi Hill Climbing: ['D', 'R']
